# RL basics: verifiable rewards, haiku edition

This tutorial uses Qwen3-4B and haiku poems to introduce the
**verifiable reward** pattern that underpins RL post-training:

1. Serve the base model.
2. Define a scoring function with a verifiable reward (syllable structure).
3. Evaluate the base model against that scorer.
4. GRPO-train the model with [slime](https://github.com/THUDM/slime) using the reward function.
5. Serve the trained checkpoint.
6. Evaluate it with the same scorer and compare.

**Why haikus?** A haiku has two attributes you can score
automatically — whether it follows the 5-7-5 syllable format
(deterministic, cheap) and whether the poem is actually good. That split between
*verifiable* and *subjective* rewards is exactly the landscape
RL post-training operates in. This tutorial covers the
verifiable half. In later tutorials, we will cover the subjective half.

The training config is intentionally tiny: 1xH100 and one
training iteration. It is a smoke run for the workflow, not a
quality run.

In [1]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@main nltk

/home/ec2-user/training-gym/.venv/bin/python: No module named uv


Note: you may need to restart the kernel to use updated packages.


In [2]:
import re

from modal_training_gym.common.dataset import HuggingFaceDataset
from modal_training_gym.common.deployment import DeploymentConfig
from modal_training_gym.common.eval import EvalConfig, EvalRowResult
from modal_training_gym.common.models import Qwen3_4B
from modal_training_gym.common.train import TrainConfig
from modal_training_gym.common.wandb import WandbConfig
from modal_training_gym.train_recipes.slime_recipe import SlimeRecipe

## Serve the base model

So, how does Qwen3-4B currently fare at writing haikus? We can
serve the base model and find out.

`DeploymentConfig.serve()` builds and deploys a vLLM app, then
returns a `ModelDeployment` with the concrete endpoint URL.

In [3]:
base_model = Qwen3_4B()
base_deployment = DeploymentConfig(
    model=base_model,
).serve()
print(base_deployment.url)

https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct


Now that the model has come alive, we can request it to write a haiku about a topic.

In [4]:
response = base_deployment.generate(
    "Write a haiku about cat.",
    chat_template_kwargs={"enable_thinking": False},
)
print(response)

Waiting for https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct (status 503)...
Waiting fo

Okay, how do we evaluate if that was a good haiku or not?
A haiku must follow the 5-7-5 syllable format.
We can count syllables using NLTK's CMU Pronouncing Dictionary
(with a regex fallback for words not in the dictionary)
and score how close each line is to its target.

In [5]:
_cmudict_cache = {}

def _get_cmudict() -> dict:
    if not _cmudict_cache:
        import nltk
        from nltk.corpus import cmudict
        nltk.download("cmudict", quiet=True)
        _cmudict_cache.update(cmudict.dict())
    return _cmudict_cache

def _count_syllables(text: str) -> int:
    cmu = _get_cmudict()
    total = 0
    for word in re.findall(r"[a-zA-Z]+", text):
        phones = cmu.get(word.lower())
        if phones:
            total += sum(p[-1].isdigit() for p in phones[0])
        else:
            count = len(re.findall(r"[aeiouy]+", word.lower()))
            if word.lower().endswith("e") and count > 1:
                count -= 1
            total += max(count, 1)
    return total

def score_haiku(response: str) -> float:
    lines = [l.strip() for l in response.strip().split("\n") if l.strip()]
    if len(lines) != 3:
        return 0.0
    total_diff = sum(
        abs(_count_syllables(line) - target)
        for line, target in zip(lines, [5, 7, 5])
    )
    return -float(total_diff)

In [6]:
response = base_deployment.generate(
    "Write a haiku about cat.",
    chat_template_kwargs={"enable_thinking": False},
)
print(response)
print(f"Score: {score_haiku(response)}")

Whiskers twitch in moonlight,  
paws pad softly on the floor—  
silence speaks in purr.
Score: -1.0


Let's also define a Haiku dataset.
Here, we use the statworx/haiku dataset from HuggingFace.
Each row has a `keywords` topic and a reference `text` haiku.
We can use this dataset to train our model.

In [7]:
class HaikuDataset(HuggingFaceDataset):
    hf_repo = "statworx/haiku"
    input_column = "keywords"
    output_column = "text"
    output_format = "jsonl"
    apply_chat_template = True
    system_prompt = (
        "You are a haiku poet. Write a haiku about the given topic. "
        "Use the 5-7-5 syllable format across three lines."
    )
    prompt_template = "Write a haiku about {input}."

train_dataset = HaikuDataset(n_rows=50)
eval_dataset = HaikuDataset(n_rows=10)

Let's take a look at the eval set. Each row has a `keywords`
topic and a reference `text` haiku.

In [8]:
df = eval_dataset.to_pandas()
print(len(df))
df.head(5)

/home/ec2-user/training-gym/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


10


,source,text,text_phonemes,keywords,keyword_phonemes,gruen_score,text_punc
0,bfbarry,Delicate savage. / You'll never hold the cinde...,deh|lax|kaxt sae|vaxjh / yuwl neh|ver hhowld d...,cinder,sihn|der,0.639071,NaN
1,bfbarry,A splash and a cry. / Words pulled from the ri...,ax splaesh aend ax kray / werdz puhld frahm dh...,the riverside,dhax rih|ver|sayd,0.563353,NaN
2,bfbarry,"Steamy, mist rising. / Rocks receiving downwar...",stiy|miy mihst ray|zaxng / raaks rax|siy|vaxng...,mist rising,mihst ray|zaxng,0.538326,NaN
3,bfbarry,You were broken glass. / But I touched you eve...,yuw wer brow|kaxn glaes / baht ay tahcht yuw i...,broken glass,brow|kaxn glaes,0.703446,NaN
4,bfbarry,Eyes dance with firelight. / The Moon and I ar...,ayz daens wihdh faxr|layt / dhax muwn aend ay ...,eyes dance,ayz daens,0.830985,NaN


Seems straightforward enough, right? How do we run an eval on our base model with this dataset?
We can transform our scoring function above into an Eval Configuration.

First, to explain, an Eval Configuration is a class that owns the model-calling loop.
The task-specific part is a scoring function passed to `.evaluate(...)`, which must
return `EvalRowResult`.

## Evaluate the base model

In [9]:
def eval_fn(_example: dict, response: str) -> EvalRowResult:
    return EvalRowResult(score=score_haiku(response), response=response)

base_eval_config = EvalConfig(
    dataset=eval_dataset,
    eval_fn=eval_fn,
    generate_kwargs={"chat_template_kwargs": {"enable_thinking": False}},
)
base_eval = base_eval_config.evaluate(base_deployment)
print(f"Average haiku score: {base_eval.mean:.1%}")

Average haiku score: -70.0%


## Train with slime

Now, let's actually train the model to write good haikus.
Here, we use the slime framework (https://github.com/THUDM/slime) on Modal.

In [10]:
async def haiku_rm(args, sample, **kwargs) -> float:
    import re
    import nltk
    from nltk.corpus import cmudict
    nltk.download("cmudict", quiet=True)
    cmu = dict(cmudict.dict())

    def _count(text):
        total = 0
        for w in re.findall(r"[a-zA-Z]+", text):
            phones = cmu.get(w.lower())
            if phones:
                total += sum(p[-1].isdigit() for p in phones[0])
            else:
                c = len(re.findall(r"[aeiouy]+", w.lower()))
                if w.lower().endswith("e") and c > 1:
                    c -= 1
                total += max(c, 1)
        return total

    lines = [l.strip() for l in sample.response.strip().split("\n") if l.strip()]
    if len(lines) != 3:
        return 0.0
    return -float(sum(abs(_count(line) - t) for line, t in zip(lines, [5, 7, 5])))

my_training_run = TrainConfig(
    model=base_model,
    dataset=train_dataset,
    recipe=SlimeRecipe(
        wandb=WandbConfig(project="gym-tutorial", group="qwen3-4b-haiku"),

        custom_rm_function=haiku_rm,

        num_rollout=10,
        save_interval=2,
        
        apply_chat_template_kwargs='{"enable_thinking": false}',

        image_run_commands=lambda image: image.run_commands(
            "uv pip install --system aiohttp nltk>=3.8.0",
            "python -c \"import nltk; nltk.download('cmudict', quiet=True)\"",
        ),
    ),
)

## Train

`TrainConfig.train()` builds the Modal app, runs training, and
returns a `TrainResult` with the run ID and checkpoint path.

In [11]:
result = my_training_run.train()

✓ Initialized. View run at 
https://modal.com/apps/modal-labs/joy-dev/ap-6Ju845Lh36yN4uM4LN0cyt
⠋ Initializing...
⠸ Creating objects...objects...
⠦ Creating objects...honPackage:modal_training_gym: Uploaded 0/8 files
⠏ Creating objects...honPackage:modal_training_gym: Uploaded 0/8 files
⠹ Creating objects...honPackage:modal_training_gym: Uploaded 19/41 files
└── ⠏ Creating mount PythonPackage:modal_training_gym: Finalizing index of 41 
⠴ Creating objects...
⠇ Creating objects...e:modal_training_gym
Building image im-MhEsLM757RdRDkAY5GuIaZm
⠙ Creating objects...
⠙ Creating objects...ackage:modal_training_gym
└── 🔨 Created mount PythonPackage:modal_training_gym

⠴ Creating objects...
⠇ Creating objects...e:modal_training_gym
⠙ Creating objects...e:modal_training_gym
⠼ Creating objects...e:modal_training_gym
m🔨 Created mount PythonPackage:modal_training_gym
=> Step 0: FROM base
⠧ Creating objects...
⠧ Creating objects...ackage:modal_training_gym
m🔨 Created mount PythonPackage:modal_training_gym
=> Step 1: COPY . /
⠇ Creating objects...
⠋ Creating objects...ackage:modal_training_gym
⠸ Creating objects...e:modal_training_gym
⠧ Creating objects...e:modal_training_gym
⠋ Creating objects...e:modal_training_gym
⠸ Creating objects...e:modal_training_gym
⠦ Creating objects...e:modal_training_gym
⠏ Creating objects...e:modal_training_gym
⠹ Creating objects...e:modal_training_gym
⠴ Creating objects...e:modal_training_gym
⠏ Creating objects...e:modal_training_gym
⠹ Creating objects...e:modal_training_gym
⠴ Creating objects...e:modal_training_gym
⠇ Creating objects...e:modal_training_gym
⠙ Creating objects...e:modal_training_gym
⠼ Creating objects...e:moda

[modal-client] 2026-05-06T16:10:07+0000 Timed out waiting for final app logs.


✓ App completed. View run at 
https://modal.com/apps/modal-labs/joy-dev/ap-6Ju845Lh36yN4uM4LN0cyt
Training complete: slime-slimerecipe-1778082242


## Serve and evaluate the trained checkpoint

The returned `TrainResult` has the checkpoint path and volume
metadata attached. Wrapping `result.model` in a `DeploymentConfig`
and calling `.serve()` deploys that checkpoint behind SGLang.

In [12]:
print(result.latest_checkpoint_path())

deployment = DeploymentConfig(
    model=result.model,
    app_name="qwen3-4b-haiku-serve",
    served_model_name="qwen3-4b-haiku",
).serve()
print(deployment.url)

/checkpoints/iter_0000009_hf
https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct


## Evaluate the trained checkpoint

Now let's run the same eval on the trained model and compare.

In [13]:
trained_eval = base_eval_config.evaluate(deployment, debug=True)
print(f"Trained haiku score: {trained_eval.mean:.1%}")

Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-

## Parameter sweeping

The reward function defines what "good" means — and different
definitions push the model in different directions. Let's define
three grading schemes and train with each:

| Scheme | Rule |
|--------|------|
| **strict** | Binary 1/0 — perfect 5-7-5 or nothing. |
| **graduated** | Negative sum of per-line syllable errors (the scorer we already wrote). |
... more

TODO: implement async so we can parallelize param sweeping. Also would be good to try to attempt partial rewards.